In [ ]:

!pip install langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers transformers torch -q

In [ ]:

from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline as hf_pipeline

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:

documents = [
    "Deep learning is a subset of machine learning that uses neural networks with many layers.",
    "Convolutional Neural Networks (CNNs) are mainly used for image recognition tasks.",
    "Recurrent Neural Networks (RNNs) are designed to work with sequential data like text and time series.",
    "Transfer learning allows a model trained on one task to be reused on a different but related task.",
    "The transformer architecture introduced the attention mechanism and is the basis for models like BERT and GPT.",
    "FAISS is a library developed by Facebook for efficient similarity search and clustering of dense vectors.",
    "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents and passes them to an LLM.",
    "LangChain is a framework that helps developers build applications powered by large language models.",
]

splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.create_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"Documents : {len(documents)}")
print(f"Chunks    : {len(chunks)}")
print("Vector store ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documents : 8
Chunks    : 8
Vector store ready!


In [ ]:

pipe = hf_pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=100,
    pad_token_id=50256
)
llm = HuggingFacePipeline(pipeline=pipe)

print("LLM loaded!")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded!


/tmp/ipykernel_5138/470763334.py:8: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:

def ask(question):

    retrieved_docs = vectorstore.similarity_search(question, k=2)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
  
    prompt = f"Answer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
    
    output = llm.invoke(prompt)
   
    answer = output.split("Answer:")[-1].strip()
    print(f"Question : {question}")
    print(f"Answer   : {answer}")
    print("Sources  :")
    for doc in retrieved_docs:
        print(f"  - {doc.page_content}")
    print("-" * 60)

In [ ]:

ask("What is RAG?")
ask("What are CNNs used for?")
ask("What is the transformer architecture?")

Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question : What is RAG?
Answer   : RAG is a language for managing a large collection of documents. This is a framework for building a collection of documents.

The word RAG comes from the Greek rēn, meaning "to be" or "to be preserved". This has many meanings, including:

to be kept secret (i.e. not taken out of the public domain)

to be preserved (i.e. not taken out of the public domain) to be kept in (in)
Sources  :
  - RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents and passes them to an LLM.
  - LangChain is a framework that helps developers build applications powered by large language models.
------------------------------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question : What are CNNs used for?
Answer   : The main purpose of CNNs is to help humans to understand the world in a more general way.

CNNs have many functions, however, some are more complex.

Let's define a few functions in Python
Sources  :
  - Convolutional Neural Networks (CNNs) are mainly used for image recognition tasks.
  - Deep learning is a subset of machine learning that uses neural networks with many layers.
------------------------------------------------------------
Question : What is the transformer architecture?
Answer   : The answer to this question is in the first
Sources  :
  - The transformer architecture introduced the attention mechanism and is the basis for models like BERT and GPT.
  - Deep learning is a subset of machine learning that uses neural networks with many layers.
------------------------------------------------------------
